In [ ]:
!apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, os
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull llama3.1:8b
!pip install -q rank_bm25 openai datasets
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"] = "ollama"
os.environ["LLM_MODEL"] = "llama3.1:8b"

from openai import OpenAI
c = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key="ollama")
r = c.chat.completions.create(model="llama3.1:8b",
    messages=[{"role": "user", "content": "Say hello in five words."}])
print("MODEL SAYS:", r.choices[0].message.content)

In [ ]:
import time
ask = lambda: c.chat.completions.create(model="llama3.1:8b",
    messages=[{"role": "user", "content": "Say hello in five words."}])
ask()                      # warm-up
t = time.time(); ask()
print("SECONDS:", round(time.time() - t, 1))
!ollama ps
!nvidia-smi --query-gpu=name,memory.used --format=csv

In [ ]:
import json, os, re, string, time, collections
import pandas as pd
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from openai import OpenAI

client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
MODEL = os.environ["LLM_MODEL"]
N, SEED, OUT = 20, 13, "/kaggle/working/runs_test.jsonl"
K, K2, M, TAU, R, T = 4, 2, 4, 0.6, 2, 3
STATS = {"calls": 0, "bad": 0}

PROMPT = """Answer the question using ONLY the passages below.
Reply with one JSON object and nothing else:
{{"answer": "<a few words, or INSUFFICIENT if the passages do not contain the answer>",
  "cites": [<ids of the passages you used>],
  "missing": "<one short search query for the fact you still need, or empty>"}}

{ctx}

Question: {q}"""

def norm(s):
    s = "".join(c for c in str(s).lower() if c not in string.punctuation)
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", s).split())

def f1(pred, gold):
    p, g = norm(pred).split(), norm(gold).split()
    common = sum((collections.Counter(p) & collections.Counter(g)).values())
    if not common: return 0.0
    pr, rc = common / len(p), common / len(g)
    return 2 * pr * rc / (pr + rc)

def insufficient(a): return norm(a) in ("", "insufficient")

def ask(q, ids, P, cost, temp=0.0):
    ctx = "\n".join(f"[{i}] {P[i]}" for i in ids)
    for attempt in range(5):
        try:
            r = client.chat.completions.create(model=MODEL, temperature=temp, max_tokens=120,
                messages=[{"role": "user", "content": PROMPT.format(ctx=ctx, q=q)}])
            break
        except Exception:
            time.sleep(2 ** attempt)
    else:
        raise RuntimeError("LLM call failed 5 times")
    cost["gen"] += 1; STATS["calls"] += 1
    cost["tok"] += r.usage.total_tokens if r.usage else 0
    try:
        o = json.loads(re.search(r"\{.*\}", r.choices[0].message.content or "", re.S).group(0))
        assert isinstance(o, dict)
    except Exception:
        o = {}; STATS["bad"] += 1
    cites = []
    for c_ in o.get("cites") or []:
        try:
            if int(c_) in ids: cites.append(int(c_))
        except Exception:
            pass
    return {"answer": str(o.get("answer") or "INSUFFICIENT").strip(),
            "cites": cites, "missing": str(o.get("missing") or "").strip()}

def retrieve(bm25, query, pool, k, cost):
    cost["ret"] += 1
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(range(len(scores)), key=lambda i: -scores[i])
    return [i for i in ranked if i not in pool][:k]

def single_pass(q, P, bm25, cost):
    pool = retrieve(bm25, q, [], K, cost)
    return ask(q, pool, P, cost)["answer"], pool, {}

def fixed_depth(q, P, bm25, cost):
    pool = retrieve(bm25, q, [], K, cost)
    o = ask(q, pool, P, cost)
    for _ in range(T - 1):
        extra = "" if insufficient(o["answer"]) else " " + o["answer"]
        pool += retrieve(bm25, q + extra, pool, K2, cost)
        o = ask(q, pool, P, cost)
    return o["answer"], pool, {}

def sc_gated(q, P, bm25, cost):
    pool, info = retrieve(bm25, q, [], K, cost), {}
    for t in range(1, T + 1):
        outs = [ask(q, pool, P, cost, temp=0.7) for _ in range(M)]
        votes = collections.Counter(norm(o["answer"]) for o in outs if not insufficient(o["answer"]))
        top, n = votes.most_common(1)[0] if votes else ("", 0)
        stop = n / M >= TAU
        if t == 1: info = {"stop1": stop, "pool1": list(pool)}
        if stop or t == T: break
        pool += retrieve(bm25, q + " " + top, pool, K2, cost)
    ans = next((o["answer"] for o in outs if norm(o["answer"]) == top), "INSUFFICIENT") if votes else "INSUFFICIENT"
    return ans, pool, info

def proposed(q, P, bm25, cost):
    pool, info = retrieve(bm25, q, [], K, cost), {}
    for t in range(1, T + 1):
        groups = [pool[j::M] for j in range(M)]
        subsets = [[i for i in pool if i not in g] for g in groups]
        outs = [ask(q, s, P, cost) for s in subsets]
        clusters = collections.defaultdict(list)
        for j, o in enumerate(outs):
            if not insufficient(o["answer"]) and o["cites"]:
                clusters[norm(o["answer"])].append(j)
        state, ans = "gap", "INSUFFICIENT"
        if clusters:
            _, A = max(clusters.items(), key=lambda kv: len(kv[1]))
            ans = outs[A[0]]["answer"]
            Pc = min((set(outs[j]["cites"]) for j in A), key=len)
            X = [j for j, s in enumerate(subsets) if Pc <= set(s)]
            AX = [j for j in A if j in X]
            kappa = len(AX) / len(X)
            if len(AX) >= 2 and kappa >= TAU: state = "corroborated"
            elif len(X) >= 2: state = "inconsistent"
            else: state = "underexposed"
        if t == 1: info = {"stop1": state == "corroborated", "pool1": list(pool)}
        if state == "corroborated": break
        if state == "inconsistent" or t == T:
            ans = ask(q, pool, P, cost)["answer"]
            break
        miss = collections.Counter(o["missing"] for o in outs if o["missing"])
        for u in [u for u, _ in miss.most_common(R)] or [q]:
            pool += retrieve(bm25, u, pool, K2, cost)
    info["state"] = state
    return ans, pool, info

def main():
    try:
        ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
    except Exception:
        ds = load_dataset("hotpot_qa", "distractor", split="validation")
    ds = ds.shuffle(seed=SEED).select(range(N))
    done = set()
    if os.path.exists(OUT):
        for line in open(OUT):
            r = json.loads(line); done.add((r["id"], r["method"]))
    methods = {"single": single_pass, "fixed": fixed_depth, "sc": sc_gated, "ours": proposed}
    with open(OUT, "a") as f:
        for n, ex in enumerate(ds, 1):
            titles = ex["context"]["title"]
            P = [t + ": " + "".join(s) for t, s in zip(titles, ex["context"]["sentences"])]
            bm25 = BM25Okapi([p.lower().split() for p in P])
            gold = set(ex["supporting_facts"]["title"])
            for name, fn in methods.items():
                if (ex["id"], name) in done: continue
                cost = {"ret": 0, "gen": 0, "tok": 0}
                ans, pool, info = fn(ex["question"], P, bm25, cost)
                row = {"id": ex["id"], "method": name, "type": ex["type"], "pred": ans, "gold": ex["answer"],
                       "em": int(norm(ans) == norm(ex["answer"])), "f1": f1(ans, ex["answer"]),
                       "recall": len(gold & {titles[i] for i in pool}) / len(gold), **cost}
                if "pool1" in info:
                    row["suff1"] = bool(gold <= {titles[i] for i in info["pool1"]})
                    row["stop1"] = bool(info["stop1"])
                    row["state"] = info.get("state")
                f.write(json.dumps(row) + "\n"); f.flush()
            print(f"question {n}/{N} done", flush=True)

t0 = time.time()
main()
df = pd.read_json(OUT, lines=True)
print("\n===== RESULTS =====")
print(df.groupby("method")[["em", "f1", "recall", "ret", "gen", "tok"]].mean().round(2))
print("minutes:", round((time.time() - t0) / 60, 1), "| calls:", STATS["calls"], "| unreadable replies:", STATS["bad"])
print(df[df.method == "ours"][["pred", "gold", "state", "gen"]].head(8).to_string())

In [ ]:
PROMPT = """Answer the question using ONLY the passages below.
If the passages do not contain every fact needed, answer INSUFFICIENT. Do not guess.
Reply with one JSON object and nothing else:
{{"answer": "<a few words, or INSUFFICIENT>",
  "cites": [<ids of the passages you used>],
  "missing": "<one short search query for the fact you still need, or empty>"}}

{ctx}

Question: {q}"""

def proposed(q, P, bm25, cost):
    pool, info = retrieve(bm25, q, [], K, cost), {}
    for t in range(1, T + 1):
        groups = [pool[j::M] for j in range(M)]
        subsets = [[i for i in pool if i not in g] for g in groups]
        outs = [ask(q, s, P, cost) for s in subsets]
        clusters = collections.defaultdict(list)
        for j, o in enumerate(outs):
            if not insufficient(o["answer"]) and o["cites"]:
                clusters[norm(o["answer"])].append(j)
        state, ans = "gap", "INSUFFICIENT"
        if clusters:
            _, A = max(clusters.items(), key=lambda kv: len(kv[1]))
            ans = outs[A[0]]["answer"]
            Pc = min((set(outs[j]["cites"]) for j in A), key=len)
            X = [j for j, s in enumerate(subsets) if Pc <= set(s)]
            AX = [j for j in A if j in X]
            kappa = len(AX) / len(X)
            if len(AX) >= 2 and kappa >= TAU: state = "corroborated"
            elif len(X) >= 2: state = "inconsistent"
            else: state = "underexposed"
        full = None
        if state == "inconsistent":                 # re-read all pages once
            full = ask(q, pool, P, cost); outs.append(full)
            if insufficient(full["answer"]): state = "gap"   # still missing: search again
            else: ans = full["answer"]
        stop = state in ("corroborated", "inconsistent")
        if t == 1: info = {"stop1": stop, "pool1": list(pool)}
        if stop: break
        if t == T:
            ans = (full or ask(q, pool, P, cost))["answer"]
            break
        miss = collections.Counter(o["missing"] for o in outs if o["missing"])
        for u in [u for u, _ in miss.most_common(R)] or [q]:
            pool += retrieve(bm25, u, pool, K2, cost)
    info["state"] = state
    return ans, pool, info

OUT = "/kaggle/working/runs_test2.jsonl"
STATS.update(calls=0, bad=0); t0 = time.time()
main()
df = pd.read_json(OUT, lines=True)
print("\n===== RESULTS 2 =====")
print(df.groupby("method")[["em", "f1", "recall", "ret", "gen", "tok"]].mean().round(2))
print("minutes:", round((time.time() - t0) / 60, 1), "| calls:", STATS["calls"], "| unreadable replies:", STATS["bad"])
print("ours states:", df[df.method == "ours"].state.value_counts().to_dict())
print(df[df.method == "ours"][["pred", "gold", "state", "ret", "gen"]].head(8).to_string())

In [ ]:
BAD_SAMPLES = []

def ask(q, ids, P, cost, temp=0.0):
    ctx = "\n".join(f"[{i}] {P[i]}" for i in ids)
    for attempt in range(5):
        try:
            r = client.chat.completions.create(model=MODEL, temperature=temp, max_tokens=200,
                messages=[{"role": "user", "content": PROMPT.format(ctx=ctx, q=q)}])
            break
        except Exception:
            time.sleep(2 ** attempt)
    else:
        raise RuntimeError("LLM call failed 5 times")
    cost["gen"] += 1; STATS["calls"] += 1
    cost["tok"] += r.usage.total_tokens if r.usage else 0
    text = r.choices[0].message.content or ""
    try:
        o = json.loads(re.search(r"\{.*\}", text, re.S).group(0))
        assert isinstance(o, dict)
    except Exception:
        o = {}; STATS["bad"] += 1; BAD_SAMPLES.append(text)
    cites = []
    for c_ in o.get("cites") or []:
        try:
            if int(c_) in ids: cites.append(int(c_))
        except Exception:
            pass
    return {"answer": str(o.get("answer") or "INSUFFICIENT").strip(),
            "cites": cites, "missing": str(o.get("missing") or "").strip()}

ND = 100
try:
    dsd = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
except Exception:
    dsd = load_dataset("hotpot_qa", "distractor", split="validation")
dsd = dsd.shuffle(seed=SEED).select(range(ND))

rows = []; STATS.update(calls=0, bad=0); t0 = time.time()
for n, ex in enumerate(dsd, 1):
    titles = ex["context"]["title"]
    P = [t + ": " + "".join(s) for t, s in zip(titles, ex["context"]["sentences"])]
    bm25 = BM25Okapi([p.lower().split() for p in P])
    gold = set(ex["supporting_facts"]["title"])
    cost = {"ret": 0, "gen": 0, "tok": 0}
    q = ex["question"]
    pool = retrieve(bm25, q, [], K, cost)
    full = ask(q, pool, P, cost)
    subs = [ask(q, [i for i in pool if i != drop], P, cost) for drop in pool]
    answered = not insufficient(full["answer"])
    same = sum(f1(o["answer"], full["answer"]) >= 0.6 for o in subs) if answered else 0
    rows.append({"suff": bool(gold <= {titles[i] for i in pool}),
                 "answered": answered,
                 "correct": f1(full["answer"], ex["answer"]) >= 0.6,
                 "n_cite": len(full["cites"]),
                 "n_sub_answered": sum(not insufficient(o["answer"]) for o in subs),
                 "n_same": same,
                 "n_crit": (len(pool) - same) if answered else 0})
    if n % 10 == 0: print(f"{n}/{ND} done", flush=True)

d = pd.DataFrame(rows); d.to_csv("/kaggle/working/diagnostic.csv", index=False)
print("\n===== DIAGNOSTIC =====")
print("pages enough (True) / not enough (False):", d.suff.value_counts().to_dict())
print(d.groupby("suff")[["answered", "correct", "n_cite", "n_sub_answered", "n_same", "n_crit"]].mean().round(2))
print(pd.crosstab(d.n_crit, d.suff))
rules = {"AI gave an answer": d.answered,
         "3+ students agree": d.n_same >= 3,
         "answer needs 2+ pages": d.answered & (d.n_crit >= 2),
         "answer cites 2+ pages": d.answered & (d.n_cite >= 2)}
for name, stop in rules.items():
    prec = d[stop].suff.mean() if stop.any() else float("nan")
    print(f"{name:22s} | stops on {stop.mean():.2f} | right to stop {prec:.2f} | catches {stop[d.suff].mean():.2f}")
print("minutes:", round((time.time() - t0) / 60, 1), "| unreadable:", STATS["bad"], "of", STATS["calls"])
for s in BAD_SAMPLES[:3]: print("UNREADABLE >>>", repr(s[:300]))

In [ ]:
d = pd.read_csv("/kaggle/working/diagnostic.csv")
print("===== DIAGNOSTIC 2 =====")
print("answers correct overall:", round(d.correct.mean(), 2))
rules = {"AI gave an answer": d.answered,
         "3+ students agree": d.n_same >= 3,
         "answer needs 2+ pages": d.answered & (d.n_crit >= 2),
         "answer cites 2+ pages": d.answered & (d.n_cite >= 2)}
for name, stop in rules.items():
    print(f"{name:22s} | fires on {stop.mean():.2f} | correct when it fires {d[stop].correct.mean():.2f}"
          f" | correct when it does not {d[~stop].correct.mean():.2f}")
print(d.groupby("n_same")[["correct", "suff"]].agg(["mean", "count"]).round(2))

In [ ]:
PROMPT = """Answer the question using ONLY the passages below.
If the passages do not contain every fact needed, answer INSUFFICIENT. Do not guess.
Reply with one JSON object and nothing else, in this form:
{{"answer": "...", "cites": [...], "missing": "..."}}
answer: a few words, or INSUFFICIENT.
cites: the numbers of the passages you used, for example [4] or [1, 6].
missing: one short search query for the fact you still need, or "" if nothing is missing.

{ctx}

Question: {q}"""

def parse(text):
    text = re.sub(r"[<>]", "", text)
    try:
        o = json.loads(re.search(r"\{.*\}", text, re.S).group(0))
        assert isinstance(o, dict)
        return o, True
    except Exception:
        a = re.search(r'"answer"\s*:\s*"([^"]*)"', text)
        c = re.search(r'"cites"\s*:\s*\[([^\]]*)\]', text)
        m = re.search(r'"missing"\s*:\s*"([^"]*)"', text)
        if not a: return {}, False
        return {"answer": a.group(1), "cites": re.findall(r"\d+", c.group(1)) if c else [],
                "missing": m.group(1) if m else ""}, True

def ask(q, ids, P, cost, temp=0.0):
    ctx = "\n".join(f"[{i}] {P[i]}" for i in ids)
    for attempt in range(5):
        try:
            r = client.chat.completions.create(model=MODEL, temperature=temp, max_tokens=200,
                messages=[{"role": "user", "content": PROMPT.format(ctx=ctx, q=q)}])
            break
        except Exception:
            time.sleep(2 ** attempt)
    else:
        raise RuntimeError("LLM call failed 5 times")
    cost["gen"] += 1; STATS["calls"] += 1
    cost["tok"] += r.usage.total_tokens if r.usage else 0
    o, ok = parse(r.choices[0].message.content or "")
    if not ok: STATS["bad"] += 1
    cites = []
    for c_ in o.get("cites") or []:
        try:
            if int(c_) in ids: cites.append(int(c_))
        except Exception:
            pass
    return {"answer": str(o.get("answer") or "INSUFFICIENT").strip(),
            "cites": cites, "missing": str(o.get("missing") or "").strip()}

def same(a, b): return (not insufficient(a)) and f1(a, b) >= 0.6

def gated(q, P, bm25, cost, vary):
    pool, rounds = retrieve(bm25, q, [], K, cost), []
    for t in range(1, T + 1):
        full = ask(q, pool, P, cost)
        answered, others = not insufficient(full["answer"]), []
        if answered and t < T:
            if vary == "evidence":   # ours: hide one group of pages each time
                others = [ask(q, [i for i in pool if i not in pool[j::M]], P, cost) for j in range(M)]
            else:                    # sc: same pages, ask again with randomness
                others = [ask(q, pool, P, cost, temp=0.7) for _ in range(M)]
        n_same = sum(same(o["answer"], full["answer"]) for o in others)
        stop = answered and t < T and n_same == M
        rounds.append({"t": t, "pool": list(pool), "answered": answered,
                       "n_same": n_same, "answer": full["answer"]})
        if stop or t == T: break
        miss = collections.Counter(o["missing"] for o in [full] + others if o["missing"])
        queries = [u for u, _ in miss.most_common(2)] or [q + (" " + full["answer"] if answered else "")]
        for u in queries:            # two new pages per round in total
            pool += retrieve(bm25, u, pool, 2 // len(queries), cost)
    return full["answer"], pool, {"rounds": rounds}

try:
    HOTPOT = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
except Exception:
    HOTPOT = load_dataset("hotpot_qa", "distractor", split="validation")

METHODS = {"single": single_pass, "fixed": fixed_depth,
           "sc": lambda q, P, b, c: gated(q, P, b, c, "sampling"),
           "ours": lambda q, P, b, c: gated(q, P, b, c, "evidence")}

def run(N, OUT):
    ds = HOTPOT.shuffle(seed=SEED).select(range(N))
    done = set()
    if os.path.exists(OUT):
        for line in open(OUT):
            r = json.loads(line); done.add((r["id"], r["method"]))
    with open(OUT, "a") as f:
        for n, ex in enumerate(ds, 1):
            titles = ex["context"]["title"]
            P = [t + ": " + "".join(s) for t, s in zip(titles, ex["context"]["sentences"])]
            bm25 = BM25Okapi([p.lower().split() for p in P])
            gold = set(ex["supporting_facts"]["title"])
            for name, fn in METHODS.items():
                if (ex["id"], name) in done: continue
                cost = {"ret": 0, "gen": 0, "tok": 0}
                ans, pool, info = fn(ex["question"], P, bm25, cost)
                row = {"id": ex["id"], "method": name, "type": ex["type"], "pred": ans, "gold": ex["answer"],
                       "em": int(norm(ans) == norm(ex["answer"])), "f1": f1(ans, ex["answer"]),
                       "recall": len(gold & {titles[i] for i in pool}) / len(gold),
                       "rounds": len(info["rounds"]) if "rounds" in info else cost["ret"], **cost}
                if "rounds" in info:
                    row["rounds_log"] = [{"t": x["t"], "answered": x["answered"], "n_same": x["n_same"],
                                          "suff": bool(gold <= {titles[i] for i in x["pool"]}),
                                          "correct": bool(f1(x["answer"], ex["answer"]) >= 0.6)}
                                         for x in info["rounds"]]
                f.write(json.dumps(row) + "\n"); f.flush()
            if n % 5 == 0: print(f"question {n}/{N} done", flush=True)

def report(path):
    df = pd.read_json(path, lines=True)
    print(df.groupby("method")[["em", "f1", "recall", "rounds", "gen", "tok"]].mean().round(2))
    for m in ["sc", "ours"]:
        r = pd.DataFrame([x for log in df[df.method == m].rounds_log for x in log])
        if len(r) == 0: continue
        r = r[(r.t < T) & r.answered]
        u = r.n_same == M
        print(f"{m:5s}| all {M} agree: {u.sum():3d} answers, {r[u].correct.mean():.2f} correct"
              f" | not all agree: {(~u).sum():3d} answers, {r[~u].correct.mean():.2f} correct")

STATS.update(calls=0, bad=0); t0 = time.time()
run(20, "/kaggle/working/final_test.jsonl")
print("\n===== RESULTS 3 =====")
report("/kaggle/working/final_test.jsonl")
print("minutes:", round((time.time() - t0) / 60, 1), "| unreadable replies:", STATS["bad"], "of", STATS["calls"])

In [ ]:
import numpy as np

if len(HOTPOT) > 1000:   # 300 fresh questions; the first 100 were used to design the method
    HOTPOT = HOTPOT.shuffle(seed=SEED).select(range(100, 400))

def report_full(path):
    df = pd.read_json(path, lines=True)
    print("questions:", df.id.nunique())
    print(df.groupby("method")[["em", "f1", "recall", "rounds", "gen", "tok"]].mean().round(3))
    print(df.pivot_table(index="method", columns="type", values="f1").round(3))
    piv, rng = df.pivot(index="id", columns="method", values="f1"), np.random.default_rng(0)
    for a, b in [("ours", "fixed"), ("ours", "sc"), ("ours", "single"), ("sc", "fixed")]:
        diff = (piv[a] - piv[b]).dropna().values
        bs = [rng.choice(diff, len(diff)).mean() for _ in range(1000)]
        print(f"F1 {a} minus {b}: {diff.mean():+.3f}  95% CI [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")
    for m in ["sc", "ours"]:
        logs = list(df[df.method == m].rounds_log)
        r1 = pd.DataFrame([log[0] for log in logs])
        a = r1[r1.answered]; u = a.n_same == M
        print(f"{m} round 1 | answered {len(a)} | all agree {u.sum()} -> {a[u].correct.mean():.3f} correct"
              f" | not all {(~u).sum()} -> {a[~u].correct.mean():.3f} correct")
        stop = r1.answered & (r1.n_same == M)
        print(f"{m} round 1 | pages enough on {r1.suff.mean():.3f} | stops on {stop.mean():.3f}"
              f" | right to stop {r1[stop].suff.mean():.3f} | catches {stop[r1.suff].mean():.3f}")
        ra = pd.DataFrame([x for log in logs for x in log]); ra = ra[(ra.t < T) & ra.answered]
        ua = ra.n_same == M
        print(f"{m} all rounds | all agree {ua.sum()} -> {ra[ua].correct.mean():.3f} correct"
              f" | not all {(~ua).sum()} -> {ra[~ua].correct.mean():.3f} correct")

STATS.update(calls=0, bad=0); t0 = time.time()
run(300, "/kaggle/working/final_300.jsonl")
print("\n===== FINAL RESULTS =====")
report_full("/kaggle/working/final_300.jsonl")
print("minutes:", round((time.time() - t0) / 60, 1), "| unreadable replies:", STATS["bad"], "of", STATS["calls"])

In [ ]:
import numpy as np

if len(HOTPOT) > 1000:   # 300 fresh questions; the first 100 were used to design the method
    HOTPOT = HOTPOT.shuffle(seed=SEED).select(range(100, 400))

def report_full(path):
    df = pd.read_json(path, lines=True)
    print("questions:", df.id.nunique())
    print(df.groupby("method")[["em", "f1", "recall", "rounds", "gen", "tok"]].mean().round(3))
    print(df.pivot_table(index="method", columns="type", values="f1").round(3))
    piv, rng = df.pivot(index="id", columns="method", values="f1"), np.random.default_rng(0)
    for a, b in [("ours", "fixed"), ("ours", "sc"), ("ours", "single"), ("sc", "fixed")]:
        diff = (piv[a] - piv[b]).dropna().values
        bs = [rng.choice(diff, len(diff)).mean() for _ in range(1000)]
        print(f"F1 {a} minus {b}: {diff.mean():+.3f}  95% CI [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")
    for m in ["sc", "ours"]:
        logs = list(df[df.method == m].rounds_log)
        r1 = pd.DataFrame([log[0] for log in logs])
        a = r1[r1.answered]; u = a.n_same == M
        print(f"{m} round 1 | answered {len(a)} | all agree {u.sum()} -> {a[u].correct.mean():.3f} correct"
              f" | not all {(~u).sum()} -> {a[~u].correct.mean():.3f} correct")
        stop = r1.answered & (r1.n_same == M)
        print(f"{m} round 1 | pages enough on {r1.suff.mean():.3f} | stops on {stop.mean():.3f}"
              f" | right to stop {r1[stop].suff.mean():.3f} | catches {stop[r1.suff].mean():.3f}")
        ra = pd.DataFrame([x for log in logs for x in log]); ra = ra[(ra.t < T) & ra.answered]
        ua = ra.n_same == M
        print(f"{m} all rounds | all agree {ua.sum()} -> {ra[ua].correct.mean():.3f} correct"
              f" | not all {(~ua).sum()} -> {ra[~ua].correct.mean():.3f} correct")

STATS.update(calls=0, bad=0); t0 = time.time()
run(300, "/kaggle/working/final_300.jsonl")
print("\n===== FINAL RESULTS =====")
report_full("/kaggle/working/final_300.jsonl")
print("minutes:", round((time.time() - t0) / 60, 1), "| unreadable replies:", STATS["bad"], "of", STATS["calls"])

In [ ]:
import numpy as np, io, contextlib, traceback

if len(HOTPOT) > 1000:   # 300 fresh questions; the first 100 were used to design the method
    HOTPOT = HOTPOT.shuffle(seed=SEED).select(range(100, 400))

def report_full(path):
    df = pd.read_json(path, lines=True)
    print("questions:", df.id.nunique())
    print(df.groupby("method")[["em", "f1", "recall", "rounds", "gen", "tok"]].mean().round(3))
    print(df.pivot_table(index="method", columns="type", values="f1").round(3))
    piv, rng = df.pivot(index="id", columns="method", values="f1"), np.random.default_rng(0)
    for a, b in [("ours", "fixed"), ("ours", "sc"), ("ours", "single"), ("sc", "fixed")]:
        diff = (piv[a] - piv[b]).dropna().values
        bs = [rng.choice(diff, len(diff)).mean() for _ in range(1000)]
        print(f"F1 {a} minus {b}: {diff.mean():+.3f}  95% CI [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")
    for m in ["sc", "ours"]:
        logs = list(df[df.method == m].rounds_log)
        r1 = pd.DataFrame([log[0] for log in logs])
        a = r1[r1.answered]; u = a.n_same == M
        print(f"{m} round 1 | answered {len(a)} | all agree {u.sum()} -> {a[u].correct.mean():.3f} correct"
              f" | not all {(~u).sum()} -> {a[~u].correct.mean():.3f} correct")
        stop = r1.answered & (r1.n_same == M)
        print(f"{m} round 1 | pages enough on {r1.suff.mean():.3f} | stops on {stop.mean():.3f}"
              f" | right to stop {r1[stop].suff.mean():.3f} | catches {stop[r1.suff].mean():.3f}")
        ra = pd.DataFrame([x for log in logs for x in log]); ra = ra[(ra.t < T) & ra.answered]
        ua = ra.n_same == M
        print(f"{m} all rounds | all agree {ua.sum()} -> {ra[ua].correct.mean():.3f} correct"
              f" | not all {(~ua).sum()} -> {ra[~ua].correct.mean():.3f} correct")

STATS.update(calls=0, bad=0); t0 = time.time()
buf = io.StringIO()
try:
    run(300, "/kaggle/working/final_300.jsonl")
except Exception:
    buf.write("RUN STOPPED EARLY\n" + traceback.format_exc() + "\n")
try:
    with contextlib.redirect_stdout(buf):
        report_full("/kaggle/working/final_300.jsonl")
except Exception:
    buf.write("\nREPORT ERROR\n" + traceback.format_exc())
text = ("===== FINAL RESULTS =====\n" + buf.getvalue() +
        f"\nminutes: {round((time.time() - t0) / 60, 1)} | unreadable replies: {STATS['bad']} of {STATS['calls']}\n")
with open("/kaggle/working/final_report.txt", "w") as f:
    f.write(text)
print("\n" + text)

In [ ]:
# BOX 10: restart the model server
import subprocess, time
subprocess.run(["pkill", "-f", "ollama"]); time.sleep(3)
subprocess.Popen(["ollama", "serve"]); time.sleep(10)
r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Say hello in five words."}])
print("MODEL SAYS:", r.choices[0].message.content)
!ollama ps

In [ ]:
# BOX 9: extras for the paper. Paste as a NEW box below the last one.
import io, contextlib, traceback, inspect, subprocess, sys
import numpy as np

EX = io.StringIO()
FINAL, WINDOW = "/kaggle/working/final_300.jsonl", "/kaggle/working/ablation_window.jsonl"

def section(title, fn):
    EX.write(f"\n===== {title} =====\n")
    try:
        with contextlib.redirect_stdout(EX):
            fn()
    except Exception:
        EX.write("ERROR\n" + traceback.format_exc())
    with open("/kaggle/working/extras_report.txt", "w") as f:   # saved after every section
        f.write(EX.getvalue())

def stopped(log):
    last = log[-1]
    return bool(last["answered"]) and last["t"] < T and last["n_same"] == M

def versions():
    import datasets, openai
    print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__,
          "| datasets", datasets.__version__, "| openai", openai.__version__)
    for cmd in (["ollama", "--version"], ["ollama", "show", MODEL], ["pip", "show", "rank_bm25"],
                ["nvidia-smi", "--query-gpu=name", "--format=csv"]):
        out = subprocess.run(cmd, capture_output=True, text=True)
        print("$", " ".join(cmd)); print((out.stdout + out.stderr).strip()[:1200])
    print("K", K, "| M", M, "| T", T, "| SEED", SEED, "| evaluation questions", len(HOTPOT))
    print("question types:", dict(collections.Counter(HOTPOT["type"])))
    print("PROMPT >>>"); print(PROMPT)
    for fn in (gated, same, ask, retrieve, fixed_depth, single_pass, run):
        print(inspect.getsource(fn))

def pilot():
    d = pd.read_csv("/kaggle/working/diagnostic.csv")
    print("questions", len(d), "| pools sufficient", int(d.suff.sum()))
    print(d.groupby("suff")[["answered", "correct", "n_cite", "n_sub_answered", "n_same", "n_crit"]].mean().round(3).to_string())
    print(d.groupby("n_same")[["correct", "suff"]].agg(["mean", "count"]).round(3).to_string())
    rules = {"gave an answer": d.answered, "3+ agree": d.n_same >= 3, "all 4 agree": d.n_same == 4,
             "needs 2+ pages": d.answered & (d.n_crit >= 2), "cites 2+ pages": d.answered & (d.n_cite >= 2)}
    for name, stop in rules.items():
        print(f"{name:15s} | stops on {stop.mean():.3f} | precision {d[stop].suff.mean():.3f}"
              f" | recall {stop[d.suff].mean():.3f} | correct when it stops {d[stop].correct.mean():.3f}")

def extra():
    df = pd.read_json(FINAL, lines=True)
    df["answered"] = [not insufficient(p) for p in df.pred]
    df["ungrounded"] = df.answered & (df.recall < 1)
    print(df.groupby("method")[["answered", "ungrounded"]].mean().round(3).to_string())
    print("share of questions ending at each round:")
    print(pd.crosstab(df.method, df.rounds, normalize="index").round(3).to_string())
    for col in ("f1", "em", "rounds", "gen"):
        print(f"mean {col} by question type:")
        print(df.pivot_table(index="method", columns="type", values=col).round(3).to_string())

def agreement():
    df = pd.read_json(FINAL, lines=True)
    for m in ("sc", "ours"):
        r1 = pd.DataFrame([log[0] for log in df[df.method == m].rounds_log])
        print(m, "| round 1 by agreement count (0 includes questions with no answer):")
        print(r1.groupby("n_same")[["correct", "suff"]].agg(["mean", "count"]).round(3).to_string())
        print(m, "| round 1, answered questions only:")
        print(r1[r1.answered].groupby("n_same")[["correct", "suff"]].agg(["mean", "count"]).round(3).to_string())

def wrong():
    df = pd.read_json(FINAL, lines=True)
    qa = {ex["id"]: (ex["question"], ex["answer"]) for ex in HOTPOT}
    ours = df[df.method == "ours"]
    w, ok = ours[ours.f1 < 0.6].copy(), ours[ours.f1 >= 0.6]
    def cat(r):
        if insufficient(r.pred): return "no answer returned"
        how = "stopped on agreement" if stopped(r.rounds_log) else "returned at last round"
        return ("wrong, gold paragraph missing, " if r.recall < 1 else "wrong, both gold present, ") + how
    w["category"] = [cat(r) for r in w.itertuples()]
    print("questions:", ours.id.nunique(), "| wrong answers of ours (F1 < 0.6):", len(w))
    print(w["category"].value_counts().to_string())
    ok_stop = [stopped(l) for l in ok.rounds_log]
    print("correct answers of ours:", len(ok), "| of which stopped on agreement:", sum(ok_stop))
    for r in ok[ok_stop].head(3).itertuples():
        print("CORRECT STOP |", qa.get(r.id, ("", ""))[0], "| gold:", qa.get(r.id, ("", ""))[1], "| pred:", r.pred, "| rounds:", r.rounds)
    s = w.sample(min(50, len(w)), random_state=0)
    out = pd.DataFrame({"id": s["id"], "type": s["type"], "category": s["category"],
                        "question": [qa.get(i, ("", ""))[0] for i in s["id"]],
                        "gold": [qa.get(i, ("", ""))[1] for i in s["id"]],
                        "pred": s["pred"], "f1": s["f1"].round(2), "gold_paragraphs_found": s["recall"],
                        "rounds": s["rounds"], "rounds_log": [json.dumps(l) for l in s["rounds_log"]]})
    out.to_csv("/kaggle/working/wrong_50.csv", index=False)
    print(out.drop(columns=["id", "rounds_log"]).head(12).to_string())

def windows(pool):                      # the original idea: M overlapping windows, each half of the pool
    n = len(pool); w = (n + 1) // 2
    return [[pool[((j * n) // M + i) % n] for i in range(w)] for j in range(M)]

def gated_window(q, P, bm25, cost):
    pool, rounds = retrieve(bm25, q, [], K, cost), []
    for t in range(1, T + 1):
        full = ask(q, pool, P, cost)
        answered, others = not insufficient(full["answer"]), []
        if answered and t < T:
            others = [ask(q, s, P, cost) for s in windows(pool)]
        n_same = sum(same(o["answer"], full["answer"]) for o in others)
        stop = answered and t < T and n_same == M
        rounds.append({"t": t, "pool": list(pool), "answered": answered, "n_same": n_same, "answer": full["answer"]})
        if stop or t == T: break
        miss = collections.Counter(o["missing"] for o in [full] + others if o["missing"])
        queries = [u for u, _ in miss.most_common(2)] or [q + (" " + full["answer"] if answered else "")]
        for u in queries:
            pool += retrieve(bm25, u, pool, 2 // len(queries), cost)
    return full["answer"], pool, {"rounds": rounds}

def ablation():
    global METHODS
    keep, METHODS = METHODS, {"window": gated_window}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run(300, WINDOW)
    finally:
        METHODS = keep
    a = pd.read_json(FINAL, lines=True); a = a[a.method == "ours"]
    b = pd.read_json(WINDOW, lines=True)
    both = pd.concat([a, b])
    print(both.groupby("method")[["em", "f1", "recall", "rounds", "gen", "tok"]].mean().round(3).to_string())
    piv = both.pivot(index="id", columns="method", values="f1").dropna()
    diff, rng = (piv["ours"] - piv["window"]).values, np.random.default_rng(0)
    bs = [rng.choice(diff, len(diff)).mean() for _ in range(1000)]
    print(f"questions in both: {len(diff)} | F1 ours minus window: {diff.mean():+.3f}"
          f"  95% CI [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")
    for m, x in (("ours", a), ("window", b)):
        r1 = pd.DataFrame([log[0] for log in x.rounds_log]); ans = r1[r1.answered]; u = ans.n_same == M
        print(f"{m} round 1 | answered {len(ans)} | all agree {int(u.sum())} -> {ans[u].correct.mean():.3f} correct,"
              f" {ans[u].suff.mean():.3f} sufficient | not all {int((~u).sum())} -> {ans[~u].correct.mean():.3f} correct")

section("VERSIONS AND SOURCE", versions)
section("PILOT (100 design questions)", pilot)
section("EXTRA METRICS", extra)
section("AGREEMENT COUNT", agreement)
section("WRONG ANSWERS", wrong)
section("ABLATION: half-pool windows", ablation)
print(EX.getvalue())

In [ ]:
# BOX 11: finish the ablation on the right 300 questions
import io, contextlib
if len(HOTPOT) > 1000:
    HOTPOT = HOTPOT.shuffle(seed=SEED).select(range(100, 400))
print("evaluation questions", len(HOTPOT), "| types", dict(collections.Counter(HOTPOT["type"])))
keep, METHODS = METHODS, {"window": gated_window}
try:
    run(300, WINDOW)          # 200 done already, about 100 to go
finally:
    METHODS = keep
a = pd.read_json(FINAL, lines=True); a = a[a.method == "ours"]
b = pd.read_json(WINDOW, lines=True); b = b[b.id.isin(set(a.id))]
both = pd.concat([a, b])
out = io.StringIO()
with contextlib.redirect_stdout(out):
    print("evaluation questions", len(HOTPOT), "| types", dict(collections.Counter(HOTPOT["type"])))
    print("window rows on evaluation questions:", b.id.nunique())
    print(both.groupby("method")[["em", "f1", "recall", "rounds", "gen", "tok"]].mean().round(3).to_string())
    piv = both.pivot(index="id", columns="method", values="f1").dropna()
    diff, rng = (piv["ours"] - piv["window"]).values, np.random.default_rng(0)
    bs = [rng.choice(diff, len(diff)).mean() for _ in range(1000)]
    print(f"questions in both: {len(diff)} | F1 ours minus window: {diff.mean():+.3f}"
          f"  95% CI [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")
    for m, x in (("ours", a), ("window", b)):
        r1 = pd.DataFrame([log[0] for log in x.rounds_log]); ans = r1[r1.answered]; u = ans.n_same == M
        print(f"{m} round 1 | answered {len(ans)} | all agree {int(u.sum())} -> {ans[u].correct.mean():.3f} correct,"
              f" {ans[u].suff.mean():.3f} sufficient | not all {int((~u).sum())} -> {ans[~u].correct.mean():.3f} correct")
        print(f"{m} ends at round:", dict(x.rounds.value_counts().sort_index()))
open("/kaggle/working/box11_report.txt", "w").write(out.getvalue()); print(out.getvalue())

In [ ]:
# BOX 12: self-contained ablation resume. Run after any restart.
import os, sys, json, re, string, time, collections, subprocess, shutil, io, contextlib
F, W = "/kaggle/working/final_300.jsonl", "/kaggle/working/ablation_window.jsonl"
print("files present:", {os.path.basename(p): os.path.exists(p) for p in (F, W)}, flush=True)
assert os.path.exists(F), "final_300.jsonl is gone: stop here and tell me"
if not shutil.which("ollama"):
    subprocess.run("apt-get install -y -qq zstd pciutils >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh", shell=True)
subprocess.run(["pkill", "-f", "ollama serve"]); time.sleep(2)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL); time.sleep(8)
subprocess.run(["ollama", "pull", "llama3.1:8b"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rank_bm25", "openai", "datasets", "pandas", "numpy"])
import numpy as np, pandas as pd
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from openai import OpenAI
client, MODEL = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama"), "llama3.1:8b"
print("MODEL SAYS:", client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Say hello in five words."}]).choices[0].message.content, flush=True)
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout, flush=True)

K, M, T, SEED, STATS = 4, 4, 3, 13, {"calls": 0, "bad": 0}
PROMPT = """Answer the question using ONLY the passages below.
If the passages do not contain every fact needed, answer INSUFFICIENT. Do not guess.
Reply with one JSON object and nothing else, in this form:
{{"answer": "...", "cites": [...], "missing": "..."}}
answer: a few words, or INSUFFICIENT.
cites: the numbers of the passages you used, for example [4] or [1, 6].
missing: one short search query for the fact you still need, or "" if nothing is missing.

{ctx}

Question: {q}"""

def norm(s):
    s = "".join(c for c in str(s).lower() if c not in string.punctuation)
    return " ".join(re.sub(r"\b(a|an|the)\b", " ", s).split())

def f1(pred, gold):
    p, g = norm(pred).split(), norm(gold).split()
    common = sum((collections.Counter(p) & collections.Counter(g)).values())
    if not common: return 0.0
    pr, rc = common / len(p), common / len(g)
    return 2 * pr * rc / (pr + rc)

def insufficient(a): return norm(a) in ("", "insufficient")

def parse(text):
    text = re.sub(r"[<>]", "", text)
    try:
        o = json.loads(re.search(r"\{.*\}", text, re.S).group(0))
        assert isinstance(o, dict)
        return o, True
    except Exception:
        a = re.search(r'"answer"\s*:\s*"([^"]*)"', text)
        c = re.search(r'"cites"\s*:\s*\[([^\]]*)\]', text)
        m = re.search(r'"missing"\s*:\s*"([^"]*)"', text)
        if not a: return {}, False
        return {"answer": a.group(1), "cites": re.findall(r"\d+", c.group(1)) if c else [],
                "missing": m.group(1) if m else ""}, True

def ask(q, ids, P, cost, temp=0.0):
    ctx = "\n".join(f"[{i}] {P[i]}" for i in ids)
    for attempt in range(5):
        try:
            r = client.chat.completions.create(model=MODEL, temperature=temp, max_tokens=200,
                messages=[{"role": "user", "content": PROMPT.format(ctx=ctx, q=q)}])
            break
        except Exception:
            time.sleep(2 ** attempt)
    else:
        raise RuntimeError("LLM call failed 5 times")
    cost["gen"] += 1; STATS["calls"] += 1
    cost["tok"] += r.usage.total_tokens if r.usage else 0
    o, ok = parse(r.choices[0].message.content or "")
    if not ok: STATS["bad"] += 1
    cites = []
    for c_ in o.get("cites") or []:
        try:
            if int(c_) in ids: cites.append(int(c_))
        except Exception:
            pass
    return {"answer": str(o.get("answer") or "INSUFFICIENT").strip(),
            "cites": cites, "missing": str(o.get("missing") or "").strip()}

def retrieve(bm25, query, pool, k, cost):
    cost["ret"] += 1
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(range(len(scores)), key=lambda i: -scores[i])
    return [i for i in ranked if i not in pool][:k]

def same(a, b): return (not insufficient(a)) and f1(a, b) >= 0.6

def windows(pool):
    n = len(pool); w = (n + 1) // 2
    return [[pool[((j * n) // M + i) % n] for i in range(w)] for j in range(M)]

def gated_window(q, P, bm25, cost):
    pool, rounds = retrieve(bm25, q, [], K, cost), []
    for t in range(1, T + 1):
        full = ask(q, pool, P, cost)
        answered, others = not insufficient(full["answer"]), []
        if answered and t < T:
            others = [ask(q, s, P, cost) for s in windows(pool)]
        n_same = sum(same(o["answer"], full["answer"]) for o in others)
        stop = answered and t < T and n_same == M
        rounds.append({"t": t, "pool": list(pool), "answered": answered, "n_same": n_same, "answer": full["answer"]})
        if stop or t == T: break
        miss = collections.Counter(o["missing"] for o in [full] + others if o["missing"])
        queries = [u for u, _ in miss.most_common(2)] or [q + (" " + full["answer"] if answered else "")]
        for u in queries:
            pool += retrieve(bm25, u, pool, 2 // len(queries), cost)
    return full["answer"], pool, {"rounds": rounds}

def run(N, OUT):
    ds = HOTPOT.shuffle(seed=SEED).select(range(N))
    done = set()
    if os.path.exists(OUT):
        for line in open(OUT):
            r = json.loads(line); done.add((r["id"], r["method"]))
    with open(OUT, "a") as f:
        for n, ex in enumerate(ds, 1):
            titles = ex["context"]["title"]
            P = [t + ": " + "".join(s) for t, s in zip(titles, ex["context"]["sentences"])]
            bm25 = BM25Okapi([p.lower().split() for p in P])
            gold = set(ex["supporting_facts"]["title"])
            for name, fn in METHODS.items():
                if (ex["id"], name) in done: continue
                cost = {"ret": 0, "gen": 0, "tok": 0}
                ans, pool, info = fn(ex["question"], P, bm25, cost)
                row = {"id": ex["id"], "method": name, "type": ex["type"], "pred": ans, "gold": ex["answer"],
                       "em": int(norm(ans) == norm(ex["answer"])), "f1": f1(ans, ex["answer"]),
                       "recall": len(gold & {titles[i] for i in pool}) / len(gold),
                       "rounds": len(info["rounds"]), **cost}
                row["rounds_log"] = [{"t": x["t"], "answered": x["answered"], "n_same": x["n_same"],
                                      "suff": bool(gold <= {titles[i] for i in x["pool"]}),
                                      "correct": bool(f1(x["answer"], ex["answer"]) >= 0.6)}
                                     for x in info["rounds"]]
                f.write(json.dumps(row) + "\n"); f.flush()
            if n % 10 == 0: print(f"question {n}/{N} done", flush=True)

try:
    HOTPOT = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
except Exception:
    HOTPOT = load_dataset("hotpot_qa", "distractor", split="validation")
HOTPOT = HOTPOT.shuffle(seed=SEED).select(range(100, 400))
METHODS = {"window": gated_window}
run(300, W)

a = pd.read_json(F, lines=True); a = a[a.method == "ours"]
b = pd.read_json(W, lines=True); b = b[b.id.isin(set(a.id))]
both = pd.concat([a, b])
out = io.StringIO()
with contextlib.redirect_stdout(out):
    print("evaluation questions", len(HOTPOT), "| types", dict(collections.Counter(HOTPOT["type"])))
    print("window rows on evaluation questions:", b.id.nunique())
    print(both.groupby("method")[["em", "f1", "recall", "rounds", "gen", "tok"]].mean().round(3).to_string())
    piv = both.pivot(index="id", columns="method", values="f1").dropna()
    diff, rng = (piv["ours"] - piv["window"]).values, np.random.default_rng(0)
    bs = [rng.choice(diff, len(diff)).mean() for _ in range(1000)]
    print(f"questions in both: {len(diff)} | F1 ours minus window: {diff.mean():+.3f}"
          f"  95% CI [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")
    for m, x in (("ours", a), ("window", b)):
        r1 = pd.DataFrame([log[0] for log in x.rounds_log]); ans = r1[r1.answered]; u = ans.n_same == M
        print(f"{m} round 1 | answered {len(ans)} | all agree {int(u.sum())} -> {ans[u].correct.mean():.3f} correct,"
              f" {ans[u].suff.mean():.3f} sufficient | not all {int((~u).sum())} -> {ans[~u].correct.mean():.3f} correct")
        print(f"{m} ends at round:", dict(x.rounds.value_counts().sort_index()))
open("/kaggle/working/box11_report.txt", "w").write(out.getvalue()); print(out.getvalue())

In [ ]:
!pip install -q datasets
from datasets import load_dataset
import collections
try: ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
except Exception: ds = load_dataset("hotpot_qa", "distractor", split="validation")
print(dict(collections.Counter(ds.shuffle(seed=13).select(range(100, 400))["type"])))